In [20]:
import pandas as pd

# Load the dataset
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Unwanted Columns List
unwanted_cols = ['Index']
train_cleaned = train.drop(columns=unwanted_cols, errors='ignore') 
test_cleaned = test.drop(columns=unwanted_cols, errors='ignore') 

# Required Feature List
print("\nThis is the NEW, CLEANED dataframe:")
train_cleaned.head()
test_cleaned.head()


This is the NEW, CLEANED dataframe:


,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,qp02z1,49,2:15,NaN,1,Not Allowed,No,NaN,NaN
1,qp02z9,49,2:15,Residential,1,Not Allowed,No,6.476213,Snowy
2,qp02yf,49,2:15,Residential,3,Allowed,Yes,22.318203,Sunny
3,qp02z6,49,2:15,Residential,2,Not Allowed,Yes,NaN,Rainy
4,qp02zd,49,2:15,Residential,1,Not Allowed,No,18.266162,Foggy


In [21]:
#Check for Any NULL Values
print(train_cleaned.isnull().sum())
print(test_cleaned.isnull().sum())

geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64
geohash             0
day                 0
timestamp           0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      1349
Weather           431
dtype: int64


In [22]:
# Create hours and minutes
train[['hour','minute']] = train['timestamp'].str.split(':', expand=True).astype(int)
test[['hour','minute']] = test['timestamp'].str.split(':', expand=True).astype(int)
train.drop('timestamp', axis=1, inplace=True)
test.drop('timestamp', axis=1, inplace=True)

In [23]:
# Find null values
print("Train null values:")
print(train.isnull().sum()[train.isnull().sum() > 0])

print("\nTest null values:")
print(test.isnull().sum()[test.isnull().sum() > 0])

Train null values:
RoadType        600
Temperature    2495
Weather         797
dtype: int64

Test null values:
RoadType        324
Temperature    1349
Weather         431
dtype: int64


In [24]:
# Fill missing values
train['RoadType'] = train['RoadType'].fillna('Unknown')
test['RoadType'] = test['RoadType'].fillna('Unknown')

train['Weather'] = train['Weather'].fillna('Unknown')
test['Weather'] = test['Weather'].fillna('Unknown')

train['Temperature'] = train['Temperature'].fillna(train['Temperature'].median())
test['Temperature'] = test['Temperature'].fillna(test['Temperature'].median())

In [25]:
#Encode Categorical Columns
categorical_cols = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather'
]
from sklearn.preprocessing import LabelEncoder

for col in categorical_cols:
    le = LabelEncoder()

    combined = pd.concat([train[col], test[col]], axis=0)

    le.fit(combined.astype(str))

    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

In [26]:
#Model Validation and Evaluation
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Split into 80% Training and 20% Validation data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42)

In [27]:
# Retrain final model on the complete dataset
final_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
final_model.fit(X, y)

# Run final predictions against the hidden Test Set
test_preds = final_model.predict(test_features)

# Format into standard submission format
submission = pd.DataFrame({'Index': test['Index'], 'demand': test_preds})
submission.to_csv('submission_final.csv', index=False)